In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import joblib

# 1. Жаңа папканың ішінде су жаңа таза база құрамыз
conn = sqlite3.connect('bank_data.db')

# 2. Модельді тез арада үйретіп алу үшін кішкене жасанды банк деректерін құрастырамыз
np.random.seed(42)
n_customers = 1000
data = {
    'credit_score': np.random.randint(400, 850, n_customers),
    'age': np.random.randint(18, 70, n_customers),
    'tenure': np.random.randint(0, 10, n_customers),
    'balance': np.round(np.random.uniform(0, 150000, n_customers), 2),
    'num_products': np.random.randint(1, 5, n_customers),
    'has_credit_card': np.random.choice([0, 1], n_customers),
    'is_active_member': np.random.choice([0, 1], n_customers),
    'estimated_salary': np.round(np.random.uniform(10000, 100000, n_customers), 2),
    'churn': np.random.choice([0, 1], n_customers, p=[0.8, 0.2])
}

df = pd.DataFrame(data)
df.to_sql('customers', conn, if_exists='replace', index=False)

# 3. Базадан деректі қайта оқып, Жасанды Интеллектіні (ИИ) үйретеміз
X = df.drop(columns=['churn'])
y = df['churn']

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)

# 4. Модельді сыртқа файл қылып шығарып сақтаймыз
joblib.dump(model, 'bank_model.pkl')
conn.close()

print("Сәтті! 'bank_data.db' және 'bank_model.pkl' файлдары папкада пайда болды!")

Сәтті! 'bank_data.db' және 'bank_model.pkl' файлдары папкада пайда болды!


In [1]:
%%writefile app.py
from fastapi import FastAPI
from pydantic import BaseModel
import joblib
import numpy as np
import logging
from datetime import datetime

# 1. Лог жүйесін реттейміз (Әрбір әрекетті "bank_server.log" деген файлға жазып отырады)
logging.basicConfig(
    filename="bank_server.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    encoding="utf-8"
)

app = FastAPI(title="Bank Churn API with Logging")
model = joblib.load("bank_model.pkl")
logging.info("Банк сервері және Жасанды Интеллект моделі сәтті іске қосылды.")

class CustomerData(BaseModel):
    credit_score: int
    age: int
    tenure: int
    balance: float
    num_products: int
    has_credit_card: int
    is_active_member: int
    estimated_salary: float

@app.get("/")
def home():
    logging.info("Басты бетке сұраныс түсті.")
    return {"message": "Банк ИИ Моделінің Сервері Қосылып Тұр!"}

@app.post("/predict")
def predict_churn(data: CustomerData):
    try:
        # Клиент мәліметі келгенін логқа жазамыз
        logging.info(
            f"Болжамға сұраныс келді: Жасы={data.age}, Кредит Скор={data.credit_score}, Баланс={data.balance}"
        )
        
        input_data = np.array([[
            data.credit_score, data.age, data.tenure, data.balance,
            data.num_products, data.has_credit_card, data.is_active_member, data.estimated_salary
        ]])
        
        prediction = model.predict(input_data)[0]
        probability = model.predict_proba(input_data)[0][1]
        result_prob = round(float(probability) * 100, 2)
        
        # ИИ-дің шешімін логқа таңбалаймыз
        logging.info(f"ИИ Болжамы сәтті аяқталды: Шешім={prediction}, Ықтималдық={result_prob}%")
        
        return {
            "churn_prediction": int(prediction),
            "churn_probability": result_prob
        }
    except Exception as e:
        # Егер кодта кенеттен қате кетсе, оны логқа ҚАТЕ деп жазады
        logging.error(
            f"Болжам жасау кезінде техникалық ҚАТЕ кетті: {str(e)}",
            exc_info=True
        )
        return {"error": "Сервер ішінде қате пайда болды"}

Overwriting app.py


In [ ]:
import sys
!{sys.executable} -m uvicorn app:app --reload